In [1]:
import numpy as np
import pandas as pd

In [2]:
from boostie.model   import boostieModel
from boostie.data    import (
    train_test_split,
)
from boostie.metrics import (
    rmse, mae, r_squared,
    log_loss, accuracy,
    confusion_matrix, precision_recall,
)

## Read Titanic CSV

In [3]:
df_raw = pd.read_csv('./resources/titanic.csv')

In [4]:
df_raw.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Define model

In [5]:
model = boostieModel(
        n_estimators  = 100,
        max_depth     = 3,
        learning_rate = 0.1,
        reg_lambda    = 1.0,
        reg_gamma     = 0.0,
        objective     = "tweedie",
    )

In [6]:
model.tweedie_power

1.5

## Preprocess cols

In [7]:
preproccess={"Embarked": "one_hot_encoding", "Sex": "one_hot_encoding"}

In [8]:
preprocessed_df = model.preprocess(df_raw, feature=preproccess, dropna=False, inplace=False)

In [9]:
preprocessed_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Embarked_S,Embarked_C,Embarked_Q,Embarked__missing,Sex_male,Sex_female
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,NaN,S,1.0,0.0,0.0,0.0,1.0,0.0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,0.0,1.0,0.0,0.0,0.0,1.0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.925,NaN,S,1.0,0.0,0.0,0.0,0.0,1.0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1,C123,S,1.0,0.0,0.0,0.0,0.0,1.0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.05,NaN,S,1.0,0.0,0.0,0.0,1.0,0.0


In [10]:
preprocessed_df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'Embarked_S',
       'Embarked_C', 'Embarked_Q', 'Embarked__missing', 'Sex_male',
       'Sex_female'],
      dtype='str')

## Train Test Split

In [11]:
TARGET = "Age"

features = ["SibSp", "Parch", 'Embarked_S',
       'Embarked_C', 'Embarked_Q', 'Embarked__missing', 'Sex_male',
       'Sex_female']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(preprocessed_df[features], preprocessed_df[TARGET], test_size=0.3, seed=69)

In [13]:
X_train

,SibSp,Parch,Embarked_S,Embarked_C,Embarked_Q,Embarked__missing,Sex_male,Sex_female
0,0,0,1.0,0.0,0.0,0.0,0.0,1.0
1,1,0,1.0,0.0,0.0,0.0,1.0,0.0
2,2,0,1.0,0.0,0.0,0.0,1.0,0.0
3,1,6,1.0,0.0,0.0,0.0,0.0,1.0
4,0,0,0.0,1.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...
618,0,0,1.0,0.0,0.0,0.0,1.0,0.0
619,0,0,1.0,0.0,0.0,0.0,1.0,0.0
620,0,0,0.0,1.0,0.0,0.0,1.0,0.0
621,0,1,1.0,0.0,0.0,0.0,0.0,1.0


In [14]:
y_train

0      31.0
1      48.0
2      28.0
3      43.0
4      35.0
       ... 
618    36.0
619    29.0
620    35.0
621    22.0
622     2.0
Length: 623, dtype: object

## Train

In [15]:
model.fit(X_train, y_train, verbose=True)

  [round   10/100]  train loss: 125.038689
  [round   20/100]  train loss: 29.287812
  [round   30/100]  train loss: 14.709466
  [round   40/100]  train loss: 11.808253
  [round   50/100]  train loss: 11.042571
  [round   60/100]  train loss: 10.800916
  [round   70/100]  train loss: 10.729374
  [round   80/100]  train loss: 10.707615
  [round   90/100]  train loss: 10.688288
  [round  100/100]  train loss: 10.662404


XGBoostModel(objective='tweedie', n_estimators=100, max_depth=3, lr=0.1, lambda=1.0, gamma=0.0)

## Evaluate

In [16]:
probs = model.predict(X_test)
print(f"  MAE  : {mae(y_test, probs):.4f}")
print(f"  RMSE  : {rmse(y_test, probs):.4f}")

  MAE  : 11.8673
  RMSE  : 15.4624


In [17]:
importances = model.feature_importances(len(model.feature_names))
for i, imp in enumerate(importances):
    bar = "█" * int(imp * 40)
    print(f"  {model.feature_names[i]}  {imp:.3f}  {bar}")

  SibSp  0.315  ████████████
  Parch  0.298  ███████████
  Embarked_S  0.042  █
  Embarked_C  0.039  █
  Embarked_Q  0.129  █████
  Embarked__missing  0.055  ██
  Sex_male  0.083  ███
  Sex_female  0.039  █
